# Results Analysis

Compare completed experiment outputs under `results/<exp_name>/`. This notebook reads saved metrics only; it does not train models.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

from src.data.load import LABELS
from src.utils.io import ensure_dir

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = ensure_dir(RESULTS_DIR / "figures/experiments")
EXPERIMENT_ORDER = [
    "exp_A_cnn_scratch",
    "exp_B_transfer",
    "exp_C_class_weight",
    "exp_D_focal_loss",
    "exp_E_augmentation",
    "exp_F_svm",
]

## Comparison Table

In [ ]:
metric_paths = sorted(RESULTS_DIR.glob("exp_*/metrics.json"))
metrics_by_exp = {}
for path in metric_paths:
    with path.open("r", encoding="utf-8") as file:
        metrics = json.load(file)
    if "per_class" not in metrics:
        continue
    metrics_by_exp[path.parent.name] = metrics

rows = []
for exp_name in EXPERIMENT_ORDER:
    if exp_name not in metrics_by_exp:
        continue
    metrics = metrics_by_exp[exp_name]
    row = {
        "experiment": exp_name,
        "accuracy": metrics.get("accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "kappa": metrics.get("cohen_kappa"),
    }
    for class_name in LABELS:
        row[f"f1_{class_name}"] = metrics["per_class"][class_name]["f1"]
    rows.append(row)

comparison_df = pd.DataFrame(rows)
if comparison_df.empty:
    raise FileNotFoundError("No metrics.json files with per_class metrics found.")

comparison_csv = RESULTS_DIR / "comparison_table.csv"
comparison_md = RESULTS_DIR / "comparison_table.md"
comparison_df.to_csv(comparison_csv, index=False)

def dataframe_to_markdown(df):
    rounded = df.copy()
    numeric_cols = rounded.select_dtypes(include="number").columns
    rounded[numeric_cols] = rounded[numeric_cols].round(4)
    header = "| " + " | ".join(rounded.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(rounded.columns)) + " |"
    body = ["| " + " | ".join(map(str, row)) + " |" for row in rounded.to_numpy()]
    return "\n".join([header, separator, *body]) + "\n"

comparison_md.write_text(dataframe_to_markdown(comparison_df), encoding="utf-8")
display(comparison_df)

## Per-Class F1 Comparison

In [ ]:
f1_columns = [f"f1_{class_name}" for class_name in LABELS]
long_df = comparison_df.melt(
    id_vars="experiment",
    value_vars=f1_columns,
    var_name="class_name",
    value_name="f1",
)
long_df["class_name"] = long_df["class_name"].str.replace("f1_", "", regex=False)

fig, ax = plt.subplots(figsize=(14, 6), constrained_layout=True)
sns.barplot(data=long_df, x="class_name", y="f1", hue="experiment", ax=ax)
ax.set_title("Per-Class F1 Across Experiments")
ax.set_xlabel("Class")
ax.set_ylabel("F1")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Experiment", bbox_to_anchor=(1.02, 1), loc="upper left")
money_chart = FIGURE_DIR / "per_class_f1_comparison.png"
fig.savefig(money_chart, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Image(filename=money_chart))

## Intervention Highlights

In [ ]:
baseline_name = "exp_A_cnn_scratch"
if baseline_name not in set(comparison_df["experiment"]):
    display(Markdown("Baseline experiment A is not available yet."))
else:
    baseline = comparison_df.set_index("experiment").loc[baseline_name]
    lines = []
    for _, row in comparison_df.iterrows():
        exp_name = row["experiment"]
        if exp_name == baseline_name:
            continue
        deltas = row[f1_columns] - baseline[f1_columns]
        best_class = deltas.idxmax().replace("f1_", "")
        worst_class = deltas.idxmin().replace("f1_", "")
        lines.append(
            f"- `{exp_name}`: largest gain `{best_class}` "
            f"({deltas.max():+.3f}); largest drop `{worst_class}` "
            f"({deltas.min():+.3f})."
        )
    if lines:
        display(Markdown("\n".join(lines)))
    else:
        display(Markdown("Only baseline A is available so far; rerun after experiments B-F complete."))

## Label Noise Revisit

`scripts/run_gradcam_analysis.py` revisits the `none` class by finding test examples labeled `none` where the best CNN predicts a defect class with confidence above 0.95. These cases are not automatically wrong labels, but they are strong candidates for manual inspection because the model sees a defect-like spatial pattern despite the `none` annotation.

In [ ]:
likely_mislabeled_path = FIGURE_DIR / "likely_mislabeled_none.png"
if likely_mislabeled_path.exists():
    display(Image(filename=likely_mislabeled_path))
else:
    display(Markdown(
        "Run `python scripts/run_gradcam_analysis.py` after the CNN experiments "
        "finish to generate `likely_mislabeled_none.png`."
    ))

The key discussion point is that `none` is not a visually homogeneous class: some samples contain enough defect-like signal that a trained CNN assigns high confidence to a defect class. In the report, treat these examples as label-noise candidates and connect them back to the EDA finding that `none` overlaps heavily with Scratch/Loc-like patterns.